# Pattern 09: Multi-hop retrieval

Follows SPEC.md §8's mandatory 8-section template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for both the embedding and
generation/hop-decision steps. Under mock, `MockLLM`'s canned hop-decision response is the same for
every call, so the number of hops actually taken isn't a meaningful signal here -- only that the
sequential hop-then-search-then-decide code path runs end to end. Section 7 is a PENDING placeholder
awaiting a real-embeddings-and-LLM run (see `tasks/todo.md`).


## Reproducibility header (SPEC.md §11)

In [1]:
import platform
import sys
import subprocess
import openai
import numpy

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: 359a825ee7b5044969f2de3f3746e79b3f067638


## Setup (loaded once, used by every section below)

In [2]:
import os
os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import get_llm, MockLLM

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()


## 1. What this pattern does

Multi-hop retrieval searches SEQUENTIALLY, not in one shot: after each search, an LLM call
(`prompts/multi_hop_prompt.txt`) decides whether another, different search is needed before there's
enough context to answer -- each hop's query depends on what the PREVIOUS hop found. This is
different from pattern 06 (multi-query), which decomposes into several sub-queries UPFRONT and runs
them all in parallel. Bounded to `MAX_HOPS` (default 2) to keep cost predictable.


## 2. When to use it

- Chained-fact ("multi-hop") questions, where the answer requires combining information from two or
  more DIFFERENT chunks that don't share much vocabulary with each other or with the question
- You expect the FIRST search often won't be enough, and want the retrieval process itself to
  recognize that and follow up, rather than relying on a single upfront decomposition
- You can afford up to `max_hops + 1` LLM calls per question


## 3. When NOT to use it

- Most of your questions are single-fact lookups -- the extra hop-decision LLM calls add cost and
  latency for no benefit when one search was always going to be enough
- Latency budget can't absorb a SEQUENTIAL chain of LLM calls (unlike multi-query's parallel
  sub-queries, each hop here must wait for the previous hop's decision before it can run)
- The hop-decision LLM call itself can produce malformed output -- `_parse_next_query()` degrades
  gracefully to stopping early rather than erroring, but that means an unreliable model silently
  loses the multi-hop benefit rather than failing loudly


## 4. Implementation

In [3]:
from recipes.multi_hop import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)


retrieved: ['arxiv:2601.00905#0', 'arxiv:2601.00100#1', 'arxiv:2601.00097#1']
answer: This is a mock response.


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="09_multi_hop",
    judges_enabled=True,
)


=== 09_multi_hop (n=18) ===
  hit@3: 0.056  [95% CI 0.000, 0.167]
  hit@10: 0.111  [95% CI 0.000, 0.278]
  mrr: 0.069  [95% CI 0.000, 0.194]
  faithfulness: 1.000  [95% CI 1.000, 1.000]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 1.000  [95% CI 1.000, 1.000]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 0.3
  p95_latency_ms: 0.4
  usd_per_query: $0.00632
  eval_usd: $0.1138


## 6. Example query walkthrough

One example per eval-set category, showing the (accumulated, across-hops) retrieved chunks and the
(mocked) final answer.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.00904#2', 'arxiv:2601.00088#0', 'arxiv:2601.00090#2']
A: This is a mock response.

--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00090#2', 'arxiv:2601.00088#0', 'arxiv:2601.00100#2']
A: This is a mock response.

--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00905#0', 'arxiv:2601.00100#1', 'arxiv:2601.00097#1']
A: This is a mock response.

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00129#1', 'arxiv:2601.00088#1', 'arxiv:2601.00121#2']
A: This is a mock response.



## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available -- see `tasks/todo.md` for tracking. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal multi-hop retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.multi_hop import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
